In [1]:
import requests
import yaml

def load_yaml_file(config_path = 'config.yaml'):
   #Carregando ficha yaml
    try:
        with open(config_path,'r') as file:
            config = yaml.safe_load(file)

            return config

    except FileNotFoundError:
        print('Arquivo de configuração não encontrado')
        raise
    except yaml.YAMLError as e:
        print(f'Erro no processamento do arquivo de configuracao {e}')
        raise

conf = load_yaml_file('config_delfos.yaml')

api_key = conf['delfos_api']['api_key']
db_config = conf['db_config']


In [3]:
import json

def get_devices_by_type(device_type_id, api_key=None):
    """
    Faz requisição GET para obter os devices de um tipo específico.
    
    Args:
        device_type_id (int): ID do tipo de dispositivo
        api_key (str, optional): Sua chave de API da Delfos
    
    Returns:
        dict: Resposta da API com os devices do tipo especificado
        None: Em caso de erro na requisição
    """
    
    # URL da API com o device_type_id
    url = f"https://api.delfos.im/solar-data/data-studio/device-types/{device_type_id}/devices"
    
    # Se não tiver API key, tenta sem autenticação primeiro
    if not api_key:
        print("Tentando sem autenticação...")
        try:
            response = requests.get(url)
            if response.status_code == 200:
                print("✅ Sucesso sem autenticação!")
                return response.json()
            else:
                print(f"❌ Status: {response.status_code}")
                print("API key pode ser necessária.")
                return None
        except Exception as e:
            print(f"❌ Erro: {e}")
            return None
    
    # Com API key, tenta diferentes formatos
    auth_formats = [
        {'api-key': api_key},
        {'X-API-Key': api_key},
        {'Authorization': api_key}
    ]
    
    for i, auth_header in enumerate(auth_formats, 1):
        headers = {
            'Content-Type': 'application/json',
            'api-key': api_key
        }
        
        try:
            print(f"Tentativa {i}: {list(auth_header.keys())[0]} = {list(auth_header.values())[0][:20]}...")
            
            response = requests.get(url, headers=headers)
            
            if response.status_code == 200:
                print(f"✅ Sucesso com formato: {list(auth_header.keys())[0]}")
                return response.json()
            elif response.status_code == 401:
                print(f"❌ 401 Unauthorized")
                continue
            elif response.status_code == 404:
                print(f"❌ 404 Not Found - Device type ID {device_type_id} não existe")
                return None
            else:
                print(f"❌ Status: {response.status_code}")
                continue
                
        except requests.exceptions.RequestException as e:
            print(f"❌ Erro na requisição: {e}")
            continue
        except json.JSONDecodeError as e:
            print(f"❌ Erro ao decodificar JSON: {e}")
            continue
    
    print("\n❌ Nenhum formato de autenticação funcionou para devices.")
    return None

In [4]:
def get_devices(api_key: str):
    # URL da API com o device_type_id
    url = f"https://api.delfos.im/solar-data/data-studio/device-types"
    
    # Com API key, tenta diferentes formatos
    auth_formats = [
        {'api-key': api_key},
        {'X-API-Key': api_key},
        {'Authorization': api_key}
    ]
    
    for i, auth_header in enumerate(auth_formats, 1):
        headers = {
            'Content-Type': 'application/json',
            'api-key': api_key
        }
        
        try:
            print(f"Tentativa {i}: {list(auth_header.keys())[0]} = {list(auth_header.values())[0][:20]}...")
            
            response = requests.get(url, headers=headers)
            
            if response.status_code == 200:
                print(f"✅ Sucesso com formato: {list(auth_header.keys())[0]}")
                return response.json()
            elif response.status_code == 401:
                print(f"❌ 401 Unauthorized")
                continue
            else:
                print(f"❌ Status: {response.status_code}")
                continue
                
        except requests.exceptions.RequestException as e:
            print(f"❌ Erro na requisição: {e}")
            continue
        except json.JSONDecodeError as e:
            print(f"❌ Erro ao decodificar JSON: {e}")
            continue
    
    print("\n❌ Nenhum formato de autenticação funcionou para devices.")
    return None

In [5]:
def get_sources(api_key: str):
    # URL da API com o device_type_id
    url = f"https://api.delfos.im/solar-data/data-studio/sources"
    
    # Com API key, tenta diferentes formatos
    auth_formats = [
        {'api-key': api_key},
        {'X-API-Key': api_key},
        {'Authorization': api_key}
    ]
    
    for i, auth_header in enumerate(auth_formats, 1):
        headers = {
            'Content-Type': 'application/json',
            'api-key': api_key
        }
        
        try:
            print(f"Tentativa {i}: {list(auth_header.keys())[0]} = {list(auth_header.values())[0][:20]}...")
            
            response = requests.get(url, headers=headers)
            
            if response.status_code == 200:
                print(f"✅ Sucesso com formato: {list(auth_header.keys())[0]}")
                return response.json()
            elif response.status_code == 401:
                print(f"❌ 401 Unauthorized")
                continue
            else:
                print(f"❌ Status: {response.status_code}")
                continue
                
        except requests.exceptions.RequestException as e:
            print(f"❌ Erro na requisição: {e}")
            continue
        except json.JSONDecodeError as e:
            print(f"❌ Erro ao decodificar JSON: {e}")
            continue
    
    print("\n❌ Nenhum formato de autenticação funcionou para devices.")
    return None

In [7]:
import pandas as pd


def load_and_prepare_data(db_config: dict, query: str) -> pd.DataFrame:
    """Carrega dados do banco com logs de monitoramento"""
    print("Carregando dados do banco de dados...")
    print(f"Query a ser executada: {query}")
    
    connection_string = (
        f"postgresql://{db_config['user']}:{db_config['password']}@"
        f"{db_config['host']}:{db_config['port']}/{db_config['dbname']}"
    )
    print(f"Conectando em: {db_config['host']}:{db_config['port']}/{db_config['dbname']}")

    try:
        engine = create_engine(connection_string)
        
        df_list = []
        chunk_count = 0
        total_rows = 0
        
        print("Iniciando leitura em chunks de 10.000 registros...")
        for chunk in pd.read_sql_query(query, engine, chunksize=10000):
            chunk_count += 1
            chunk_rows = len(chunk)
            total_rows += chunk_rows
            df_list.append(chunk)
            print(f"Chunk {chunk_count}: {chunk_rows} registros carregados")

        final_df = pd.concat(df_list, ignore_index=True)
        print(f"Carregamento concluído: {total_rows} registros em {chunk_count} chunks")
        print(f"DataFrame final shape: {final_df.shape}")
        
        return final_df

    except Exception as e:
        print(f"Erro ao carregar dados do banco: {e}")
        print(f"Detalhes da conexão: {db_config}")
        raise

inv_ids = load_and_prepare_data(
    db_config,
    'SELECT * FROM "Inversores_ids"'
)

inv_ids['name'] = inv_ids['name'].str.split('-').str[0].str.strip()

inv_ids[inv_ids['name'] == 'Massapê II']

Carregando dados do banco de dados...
Query a ser executada: SELECT * FROM "Inversores_ids"
Conectando em: 24.199.119.9:5432/variaveis_delfos
Erro ao carregar dados do banco: name 'create_engine' is not defined
Detalhes da conexão: {'host': '24.199.119.9', 'port': 5432, 'dbname': 'variaveis_delfos', 'user': 'pedro_arthur', 'password': 'goener'}


NameError: name 'create_engine' is not defined

In [6]:
import requests as req
import json

api_key = 'PassYourAPISecretKeyHere'

# Timeseries
"""
data =  {
    "device_ids": [4, 5, 6, 7, 8, 9, 10, 11],
    "variable_ids": ['signal-131'],
    "start_time": "2024-03-10 00:20:00",
    "end_time": "2025-05-11 19:50:00",
    "aggregate": "1 min"
}
"""
data = {"device_type_id": 2}

headers = {
    "API-Key": "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP",
    "Content-Type": "application/json"
}

url = f'https://api.delfos.im/solar-data/data-studio/taxonomy/signal'
r = req.post(url, data=json.dumps(data), headers=headers)
r.json()



[{'id': 1,
  'name': 'DC Current',
  'device_type_id': 6,
  'device_type': 'String',
  'source_id': 1,
  'source': 'SCADA',
  'description': None,
  'unit': 'A',
  'data_type': 'double precision'},
 {'id': 2,
  'name': 'DC Power',
  'device_type_id': 6,
  'device_type': 'String',
  'source_id': 1,
  'source': 'SCADA',
  'description': None,
  'unit': 'kW',
  'data_type': 'double precision'},
 {'id': 3,
  'name': 'DC Voltage',
  'device_type_id': 6,
  'device_type': 'String',
  'source_id': 1,
  'source': 'SCADA',
  'description': None,
  'unit': 'V',
  'data_type': 'double precision'},
 {'id': 332,
  'name': 'DC Current',
  'device_type_id': 2,
  'device_type': 'Inverter',
  'source_id': 1,
  'source': 'SCADA',
  'description': None,
  'unit': 'A',
  'data_type': 'double precision'},
 {'id': 333,
  'name': 'Generated Energy',
  'device_type_id': 2,
  'device_type': 'Inverter',
  'source_id': 1,
  'source': 'SCADA',
  'description': None,
  'unit': 'kWh',
  'data_type': 'double precisio

  {
    "id": 129,
    
    "name": "Daily Generated Energy",
    
    "device_type_id": 2,
    
    "device_type": "Inverter",
    
    "source_id": 1,
    
    "source": "SCADA",
    
    "description": null,
    
    "unit": "kWh",
    
    "data_type": "double precision"
  }


In [7]:
def extract_data_batch(url, headers, device_ids, variable_ids, start_time, end_time):
    """Extrai um lote específico de dados da API"""
    
    r_config = {
        "device_ids": device_ids,
        "variable_ids": variable_ids,
        "start_time": start_time,
        "end_time": end_time,
        "aggregate": "5 min"
    }
    
    r = req.post(url, headers=headers, data=json.dumps(r_config), timeout=120)
    
    if r.status_code != 200:
        error_msg = r.text[:200] if r.text else "Sem detalhes do erro"
        display(f"❌ Erro na requisição: Status code {r.status_code}, Detalhes: {error_msg}")
        raise req.exceptions.RequestException(f"Status code: {r.status_code}, Erro: {error_msg}")
    
    response = r.json()
    
    # Verificar estrutura da resposta
    if 'sample_time' not in response or 'variables' not in response:
        raise KeyError("Estrutura de resposta inválida")
    
    # Transformar em DataFrame
    df = pd.DataFrame({'tempo': response['sample_time']})
    for var in response['variables']:
        device_name = var['device']['name']
        df[device_name] = var['values']
    
    return df


In [8]:
from datetime import datetime, timedelta
import requests as req
import pandas as pd
import json

def extract_data(headers, device_ids, variable_ids, device_type_id, start_time=None, end_time=None):
    """Extrai dados da API Delfos com processamento em lotes"""

    # Configurar tempos padrão
    if end_time is None:
        end_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    if start_time is None:
        start_time = (datetime.now() - timedelta(days=5)).strftime('%Y-%m-%d %H:%M:%S')
    
    # Calcular período e verificar se precisa dividir em lotes
    start_dt = datetime.strptime(start_time, '%Y-%m-%d %H:%M:%S')
    end_dt = datetime.strptime(end_time, '%Y-%m-%d %H:%M:%S')
    total_days = (end_dt - start_dt).days

    # Definir tamanho do lote (máximo 7 dias para evitar erro "Too much data")
    batch_days = 7
    url = f'https://api.delfos.im/solar-data/data-studio/device-types/{device_type_id}/timeseries'
    
    if total_days <= batch_days:
        try:
            df = extract_data_batch(url=url, headers=headers, device_ids=device_ids, variable_ids=variable_ids, start_time=start_time, end_time=end_time)
            
            # Verificar dados nulos
            null_counts = df.isnull().sum()
            if null_counts.sum() > 0:
                for col, count in null_counts.items():
                    if count > 0:
                        pass  # Aqui poderia tratar os nulos se necessário

            return df
            
        except Exception as e:
            raise
    
    else:
        # Dividir em lotes
        batch_list = []
        current_start = start_dt
        batch_count = 0
        successful_batches = 0
        failed_batches = 0
        
        while current_start < end_dt:
            current_end = min(current_start + timedelta(days=batch_days), end_dt)
            batch_count += 1
            
            batch_start_str = current_start.strftime('%Y-%m-%d %H:%M:%S')
            batch_end_str = current_end.strftime('%Y-%m-%d %H:%M:%S')
            
            try:
                batch_df = extract_data_batch(url=url, headers=headers, device_ids=device_ids, variable_ids=variable_ids, start_time=batch_start_str, end_time=batch_end_str)
                
                if not batch_df.empty:
                    batch_df.dropna(inplace=True)  # Remover nulos do lote
                    batch_list.append(batch_df)
                    successful_batches += 1
                else:
                    pass  # Pode-se adicionar tratamento para lotes vazios
                
            except Exception as e:
                failed_batches += 1
                if failed_batches > 3 and successful_batches == 0:
                    raise
            
            current_start = current_end
        
        # Consolidar lotes
        if batch_list:
            df_final = pd.concat(batch_list, ignore_index=True)
            
            # Remover duplicatas baseadas na coluna 'tempo'
            df_final = df_final.drop_duplicates(subset=['tempo'], keep='first')
            
            # Ordenar por tempo
            df_final = df_final.sort_values('tempo').reset_index(drop=True)
            
            # Verificar dados nulos
            null_counts = df_final.isnull().sum()
            if null_counts.sum() > 0:
                for col, count in null_counts.items():
                    if count > 0:
                        pass  # Aqui poderia tratar os nulos se necessário
            
            return df_final
        
        else:
            raise Exception("Falha em todos os lotes de extração")


In [14]:
headers = {
            "API-Key": "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP",
            "Content-Type": "application/json"
        }

device_ids = [4, 5, 6, 7, 8, 9, 10, 11]

df_aquiraz = extract_data(headers, device_ids, ['signal-129'], '2025-01-01 00:00:00')

df_aquiraz.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125747 entries, 0 to 125746
Data columns (total 9 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   tempo                     125747 non-null  object 
 1   Aquiraz - UFV 01 - INV01  117350 non-null  float64
 2   Aquiraz - UFV 03 - INV01  117347 non-null  float64
 3   Aquiraz - UFV 01 - INV03  117353 non-null  float64
 4   Aquiraz - UFV 02 - INV03  117352 non-null  float64
 5   Aquiraz - UFV 03 - INV02  117350 non-null  float64
 6   Aquiraz - UFV 02 - INV02  117347 non-null  float64
 7   Aquiraz - UFV 02 - INV01  117351 non-null  float64
 8   Aquiraz - UFV 01 - INV02  117348 non-null  float64
dtypes: float64(8), object(1)
memory usage: 8.6+ MB


C:\Users\usuario\AppData\Local\Temp\ipykernel_2184\566682917.py:73: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


In [15]:
df_aquiraz.head()

,tempo,Aquiraz - UFV 01 - INV01,Aquiraz - UFV 03 - INV01,Aquiraz - UFV 01 - INV03,Aquiraz - UFV 02 - INV03,Aquiraz - UFV 03 - INV02,Aquiraz - UFV 02 - INV02,Aquiraz - UFV 02 - INV01,Aquiraz - UFV 01 - INV02
0,2025-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df_aquiraz[(df_aquiraz['tempo'] > '2025-11-12 06:00:00') & (df_aquiraz['tempo'] < '2025-11-12 20:00:00')]

,tempo,Aquiraz - UFV 01 - INV01,Aquiraz - UFV 03 - INV01,Aquiraz - UFV 01 - INV03,Aquiraz - UFV 02 - INV03,Aquiraz - UFV 03 - INV02,Aquiraz - UFV 02 - INV02,Aquiraz - UFV 02 - INV01,Aquiraz - UFV 01 - INV02
90792,2025-11-12 06:05:00,132.915410,0.00,633.463308,529.977658,456.939008,420.013548,0.0,0.000000
90793,2025-11-12 06:10:00,133.370921,0.00,634.262543,530.736893,0.685492,420.350534,0.0,0.000000
90794,2025-11-12 06:15:00,133.905392,0.00,635.204192,531.634544,1.681280,420.803536,0.0,0.000000
90795,2025-11-12 06:20:00,134.427122,0.00,636.115375,532.457645,2.598271,421.194763,0.0,0.399345
90796,2025-11-12 06:25:00,135.628700,0.00,638.209236,534.292917,4.685236,422.475120,0.0,2.505501
...,...,...,...,...,...,...,...,...,...
90954,2025-11-12 19:35:00,129.630000,764.18,788.800000,572.730000,693.860000,430.700000,0.0,640.120000
90955,2025-11-12 19:40:00,129.630000,764.18,788.800000,572.730000,693.860000,430.700000,0.0,640.120000
90956,2025-11-12 19:45:00,129.630000,764.18,788.800000,572.730000,693.860000,430.700000,0.0,640.120000
90957,2025-11-12 19:50:00,129.630000,764.18,788.800000,572.730000,693.860000,430.700000,0.0,640.120000


In [19]:
def standart_columns(df, usina):
    """Padroniza nomes das colunas por usina com logs detalhados"""
    print(f"Padronizando colunas para usina: {usina}")
    print(f"Colunas antes da padronização: {list(df.columns)}")
    
    original_columns = df.columns.tolist()
    
    if usina == 'aqz01':
        column_mapping = {
            'Aquiraz - UFV 01 - INV01': 'energia_inversor_01', 
            'Aquiraz - UFV 01 - INV02': 'energia_inversor_02',
            'Aquiraz - UFV 01 - INV03': 'energia_inversor_03', 
            'Aquiraz - UFV 02 - INV01': 'energia_inversor_04',
            'Aquiraz - UFV 02 - INV02': 'energia_inversor_05', 
            'Aquiraz - UFV 02 - INV03': 'energia_inversor_06',
            'Aquiraz - UFV 03 - INV01': 'energia_inversor_07', 
            'Aquiraz - UFV 03 - INV02': 'energia_inversor_08'
        }
        df = df.rename(columns=column_mapping)
    
    elif usina == 'utxt01':
        column_mapping = {
            'Unitêxtil - UFV 09 - INV01':'energia_inversor_01','Unitêxtil - UFV 09 - INV02':'energia_inversor_02',
            'Unitêxtil - UFV 09 - INV03':'energia_inversor_03','Unitêxtil - UFV 10 - INV01':'energia_inversor_04',
            'Unitêxtil - UFV 10 - INV02':'energia_inversor_05','Unitêxtil - UFV 10 - INV03':'energia_inversor_06',
            'Unitêxtil - UFV 11 - INV01':'energia_inversor_07','Unitêxtil - UFV 11 - INV02':'energia_inversor_08',
            'Unitêxtil - UFV 11 - INV03':'energia_inversor_09','Unitêxtil - UFV 12 - INV01':'energia_inversor_10',
            'Unitêxtil - UFV 12 - INV02':'energia_inversor_11','Unitêxtil - UFV 12 - INV03':'energia_inversor_12',
            'Unitêxtil - UFV 13 - INV01':'energia_inversor_13','Unitêxtil - UFV 13 - INV02':'energia_inversor_14',
            'Unitêxtil - UFV 13 - INV03':'energia_inversor_15','Unitêxtil - UFV 14 - INV01':'energia_inversor_16',
            'Unitêxtil - UFV 14 - INV02':'energia_inversor_17'
        }
        df = df.rename(columns=column_mapping)
    
    elif usina == 'msp02':
        column_mapping = {
            'Massapê II - INV 02': 'energia_inversor_02', 'Massapê II - INV 01': 'energia_inversor_01',
            'Massapê II - INV 03': 'energia_inversor_03', 'Massapê II - INV 04': 'energia_inversor_04',
            'Massapê II - INV 05': 'energia_inversor_05', 'Massapê II - INV 06': 'energia_inversor_06',
            'Massapê II - INV 07': 'energia_inversor_07', 'Massapê II - INV 08': 'energia_inversor_08',
            'Massapê II - INV 09': 'energia_inversor_09', 'Massapê II - INV 10': 'energia_inversor_10',
            'Massapê II - INV 11': 'energia_inversor_11', 'Massapê II - INV 12': 'energia_inversor_12',
            'Massapê II - INV 13': 'energia_inversor_13'
        }
        df = df.rename(columns=column_mapping)
    
    elif usina == 'bbr01':
        column_mapping = {
            'Beberibe - INV01': 'energia_inversor_01', 'Beberibe - INV02': 'energia_inversor_02',
            'Beberibe - INV03': 'energia_inversor_03', 'Beberibe - INV04': 'energia_inversor_04',
            'Beberibe - INV05': 'energia_inversor_05', 'Beberibe - INV06': 'energia_inversor_06',
            'Beberibe - INV07': 'energia_inversor_07', 'Beberibe - INV08': 'energia_inversor_08',
            'Beberibe - INV09': 'energia_inversor_09', 'Beberibe - INV10': 'energia_inversor_10'
        }
        df = df.rename(columns=column_mapping)

    else:
        print(f"Usina '{usina}' não reconhecida, mantendo colunas originais")
    
    return df
    


In [20]:
df_aquiraz = standart_columns(df_aquiraz, 'aqz01')

df_aquiraz.info()

Padronizando colunas para usina: aqz01
Colunas antes da padronização: ['tempo', 'Aquiraz - UFV 01 - INV01', 'Aquiraz - UFV 03 - INV01', 'Aquiraz - UFV 01 - INV03', 'Aquiraz - UFV 02 - INV03', 'Aquiraz - UFV 03 - INV02', 'Aquiraz - UFV 02 - INV02', 'Aquiraz - UFV 02 - INV01', 'Aquiraz - UFV 01 - INV02']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125747 entries, 0 to 125746
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   tempo                125747 non-null  object 
 1   energia_inversor_01  117350 non-null  float64
 2   energia_inversor_07  117347 non-null  float64
 3   energia_inversor_03  117353 non-null  float64
 4   energia_inversor_06  117352 non-null  float64
 5   energia_inversor_08  117350 non-null  float64
 6   energia_inversor_05  117347 non-null  float64
 7   energia_inversor_04  117351 non-null  float64
 8   energia_inversor_02  117348 non-null  float64
dtypes: float64(8), objec

In [21]:
df_aquiraz.head()

,tempo,energia_inversor_01,energia_inversor_07,energia_inversor_03,energia_inversor_06,energia_inversor_08,energia_inversor_05,energia_inversor_04,energia_inversor_02
0,2025-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
device_ids = [218, 219, 220, 221, 222, 223, 224, 225,
              226, 227, 228, 229, 230, 231, 232, 233]

df_unitextil = extract_data(headers, device_ids, ['signal-129'], '2025-01-01 00:00:00')

df_unitextil = standart_columns(df_unitextil, 'utxt01')

df_unitextil.info()

Padronizando colunas para usina: utxt01
Colunas antes da padronização: ['tempo', 'Unitêxtil - UFV 12 - INV03', 'Unitêxtil - UFV 10 - INV02', 'Unitêxtil - UFV 14 - INV02', 'Unitêxtil - UFV 13 - INV01', 'Unitêxtil - UFV 13 - INV02', 'Unitêxtil - UFV 11 - INV01', 'Unitêxtil - UFV 13 - INV03', 'Unitêxtil - UFV 14 - INV01', 'Unitêxtil - UFV 12 - INV02', 'Unitêxtil - UFV 12 - INV01', 'Unitêxtil - UFV 11 - INV02', 'Unitêxtil - UFV 10 - INV03', 'Unitêxtil - UFV 10 - INV01', 'Unitêxtil - UFV 09 - INV03', 'Unitêxtil - UFV 09 - INV02', 'Unitêxtil - UFV 09 - INV01']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125750 entries, 0 to 125749
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   tempo                125750 non-null  object 
 1   energia_inversor_12  110416 non-null  float64
 2   energia_inversor_05  110450 non-null  float64
 3   energia_inversor_17  106196 non-null  float64
 4   energia_inversor_

C:\Users\usuario\AppData\Local\Temp\ipykernel_2184\566682917.py:73: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


In [23]:
df_unitextil.head()

,tempo,energia_inversor_12,energia_inversor_05,energia_inversor_17,energia_inversor_13,energia_inversor_14,energia_inversor_07,energia_inversor_15,energia_inversor_16,energia_inversor_11,energia_inversor_10,energia_inversor_08,energia_inversor_06,energia_inversor_04,energia_inversor_03,energia_inversor_02,energia_inversor_01
0,2025-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
df_unitextil[(df_unitextil['tempo'] > '2025-11-12 05:00:00') & (df_unitextil['tempo'] < '2025-11-12 20:00:00')]

,tempo,energia_inversor_12,energia_inversor_05,energia_inversor_17,energia_inversor_13,energia_inversor_14,energia_inversor_07,energia_inversor_15,energia_inversor_16,energia_inversor_11,energia_inversor_10,energia_inversor_08,energia_inversor_06,energia_inversor_04,energia_inversor_03,energia_inversor_02,energia_inversor_01
90780,2025-11-12 05:05:00,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,47820.000000,0.000000,0.000000,0.000000,0.000000
90781,2025-11-12 05:10:00,0.776133,0.776133,0.000000,0.0,0.0,0.776133,0.0,0.776133,0.776133,0.000000,0.776133,47820.776133,0.000000,0.776133,0.776133,0.000000
90782,2025-11-12 05:15:00,12.776133,12.776133,2.776133,0.0,0.0,12.776133,0.0,12.776133,12.776133,2.776133,12.776133,47832.776133,2.776133,12.776133,12.776133,2.776133
90783,2025-11-12 05:20:00,25.551400,29.551433,14.775700,0.0,0.0,25.551400,0.0,29.551433,25.551400,15.551400,25.551400,47845.551400,14.775700,25.551400,25.551400,10.775700
90784,2025-11-12 05:25:00,49.551400,49.551400,26.775700,0.0,0.0,42.775700,0.0,49.551400,49.551400,32.775700,42.775700,47869.551400,29.551400,49.551400,49.551400,26.775700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90954,2025-11-12 19:35:00,84030.000000,82120.000000,44500.000000,42060.0,68300.0,76300.000000,68000.0,80800.000000,82800.000000,55650.000000,75630.000000,79660.000000,44070.000000,82680.000000,81500.000000,0.000000
90955,2025-11-12 19:40:00,84030.000000,82120.000000,44500.000000,42060.0,68300.0,76300.000000,68000.0,80800.000000,82800.000000,55650.000000,75630.000000,79660.000000,44070.000000,82680.000000,81500.000000,0.000000
90956,2025-11-12 19:45:00,84030.000000,82120.000000,44500.000000,42060.0,68300.0,76300.000000,68000.0,80800.000000,82800.000000,55650.000000,75630.000000,79660.000000,44070.000000,82680.000000,81500.000000,0.000000
90957,2025-11-12 19:50:00,84030.000000,82120.000000,44500.000000,42060.0,68300.0,76300.000000,68000.0,80800.000000,82800.000000,55650.000000,75630.000000,79660.000000,44070.000000,82680.000000,81500.000000,0.000000


In [27]:
df_unitextil['tempo'] = pd.to_datetime(df_unitextil['tempo'])

df_unitextil.describe()

,tempo,energia_inversor_12,energia_inversor_05,energia_inversor_17,energia_inversor_13,energia_inversor_14,energia_inversor_07,energia_inversor_15,energia_inversor_16,energia_inversor_11,energia_inversor_10,energia_inversor_08,energia_inversor_06,energia_inversor_04,energia_inversor_03,energia_inversor_02,energia_inversor_01
count,125750,110416.000000,110450.000000,106196.000000,110456.000000,110447.000000,108229.000000,110440.000000,104270.000000,110405.000000,109069.000000,110459.000000,110462.000000,110468.000000,110467.000000,110472.000000,110470.00000
mean,2025-08-07 07:37:30.000000512,45077.393509,52406.743731,18222.338204,32009.331804,50185.817647,50363.500701,52202.554661,43002.618245,54751.894433,34620.315973,49591.401439,48418.564855,28782.004760,42257.012822,42414.454823,7806.19028
min,2025-01-01 00:05:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2025-04-20 03:51:15,7334.456325,20347.098083,152.917075,11692.861633,16754.281017,20668.835633,24171.531458,4066.721642,28157.346267,15330.000000,18085.713317,12668.726800,11051.609192,0.000000,0.000000,0.00000
50%,2025-08-07 07:37:30,46140.000000,64630.000000,13150.000000,38780.000000,58354.225000,61440.000000,57830.000000,44773.238300,64590.055000,37628.174867,60486.394333,59738.757333,34320.000000,46102.570333,46745.061950,0.00000
75%,2025-11-24 11:23:45,79760.000000,80220.000000,35320.000000,49594.613775,80048.443500,76060.000000,80086.703267,75610.000000,80727.425800,53280.000000,76204.140933,78324.973483,43980.000000,78030.000000,77800.000000,7586.61270
max,2026-03-13 15:10:00,96970.000000,97770.000000,79910.000000,99910.000000,97940.000000,165940.000000,96760.000000,166330.000000,96640.000000,67300.000000,98720.000000,97480.000000,100030.000000,96970.000000,182300.000000,59690.00000
std,NaN,34979.865837,32335.737076,17087.694348,20121.254338,32914.191548,30309.683836,31090.457677,33775.608260,30614.229921,21009.935825,30493.449170,33088.913817,18327.747028,36496.851318,36754.227521,14664.22264


In [28]:
device_ids = [523, 524, 525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535]

df_massape = extract_data(headers, device_ids, ['signal-129'], '2025-01-01 00:00:00')

df_massape = standart_columns(df_massape, 'msp02')

df_massape.info()

C:\Users\usuario\AppData\Local\Temp\ipykernel_2184\566682917.py:73: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


Padronizando colunas para usina: msp02
Colunas antes da padronização: ['tempo', 'Massapê II - INV 02', 'Massapê II - INV 01', 'Massapê II - INV 03', 'Massapê II - INV 07', 'Massapê II - INV 09', 'Massapê II - INV 05', 'Massapê II - INV 06', 'Massapê II - INV 13', 'Massapê II - INV 08', 'Massapê II - INV 12', 'Massapê II - INV 10', 'Massapê II - INV 11', 'Massapê II - INV 04']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125752 entries, 0 to 125751
Data columns (total 14 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   tempo                125752 non-null  object 
 1   energia_inversor_02  100176 non-null  float64
 2   energia_inversor_01  100165 non-null  float64
 3   energia_inversor_03  100172 non-null  float64
 4   energia_inversor_07  100171 non-null  float64
 5   energia_inversor_09  100170 non-null  float64
 6   energia_inversor_05  100175 non-null  float64
 7   energia_inversor_06  100168 non-null  float64


In [29]:
device_ids = [803, 804, 805, 806, 807, 808, 809, 810, 811, 812]

df_beberibe = extract_data(headers, device_ids, ['signal-129'], '2025-01-01 00:00:00')

df_beberibe = standart_columns(df_beberibe, 'msp02')

df_beberibe.info()

C:\Users\usuario\AppData\Local\Temp\ipykernel_2184\566682917.py:73: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


Padronizando colunas para usina: msp02
Colunas antes da padronização: ['tempo', 'Beberibe - INV10', 'Beberibe - INV09', 'Beberibe - INV08', 'Beberibe - INV04', 'Beberibe - INV02', 'Beberibe - INV07', 'Beberibe - INV06', 'Beberibe - INV05', 'Beberibe - INV03', 'Beberibe - INV01']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125753 entries, 0 to 125752
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   tempo             125753 non-null  object 
 1   Beberibe - INV10  89625 non-null   float64
 2   Beberibe - INV09  89624 non-null   float64
 3   Beberibe - INV08  89671 non-null   float64
 4   Beberibe - INV04  89626 non-null   float64
 5   Beberibe - INV02  89673 non-null   float64
 6   Beberibe - INV07  89600 non-null   float64
 7   Beberibe - INV06  89671 non-null   float64
 8   Beberibe - INV05  89670 non-null   float64
 9   Beberibe - INV03  89675 non-null   float64
 10  Beberibe - INV01  89675 non-

In [30]:
import pandas as pd

with pd.ExcelWriter('dados_usinas.xlsx') as writer:
    df_aquiraz.to_excel(writer, sheet_name='Aquiraz', index=False)
    df_unitextil.to_excel(writer, sheet_name='Unitextil', index=False)
    df_massape.to_excel(writer, sheet_name='Massape', index=False)
    df_beberibe.to_excel(writer, sheet_name='Beberibe', index=False)

{
 'id': 33,
 'name': 'Tilted Irradiance',
 'device_type_id': 2,
 'device_type': 'Inverter',
 'source_id': 4,
 'source': 'Calc - Physical Model',
 'description': 'Irradiância inclinada média do inversor. Essa variável é associada conforme a irradiância inclinada do solar field',
 'unit': 'W/m²',
 'data_type': 'double precision'
}

{
 'id': 34,
 'name': 'Tilted Irradiance - Temperature Corrected',
 'device_type_id': 2,
 'device_type': 'Inverter',
 'source_id': 4,
 'source': 'Calc - Physical Model',
 'description': 'Irradiância inclinada corrigida pela temperatura dos módulos',
 'unit': 'W/m²',
 'data_type': 'double precision'
}

{
 'id': 39,
 'name': 'Tilted Irradiance',
 'device_type_id': 10,
 'device_type': 'Solar Field',
 'source_id': 4,
 'source': 'Calc - Physical Model',
 'description': 'Irradiância inclinada média na superfície do solar field',
 'unit': 'W/m²',
 'data_type': 'double precision'
}

{
 'id': 45,
 'name': 'Inverter Tilted Irradiance',
 'device_type_id': 6,
 'device_type': 'String',
 'source_id': 3,
 'source': 'Calculated',
 'description': None,
 'unit': '-',
 'data_type': 'double precision'
}

{
 'id': 49,
 'name': 'measured_tilted_irradiance',
 'device_type_id': 10,
 'device_type': 'Solar Field',
 'source_id': 3,
 'source': 'Calculated',
 'description': None,
 'unit': 'W/m²',
 'data_type': 'double precision'
}

{
 'id': 50,
 'name': 'transposed_tilted_irradiance',
 'device_type_id': 10,
 'device_type': 'Solar Field',
 'source_id': 3,
 'source': 'Calculated',
 'description': None,
 'unit': 'W/m²',
 'data_type': 'double precision'
}

{
 'id': 300,
 'name': 'Tilted Irradiance',
 'device_type_id': 3,
 'device_type': 'Meteo',
 'source_id': 1,
 'source': 'SCADA',
 'description': None,
 'unit': 'W/m²',
 'data_type': 'double precision'
}

In [16]:
headers = {
            "API-Key": "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP",
            "Content-Type": "application/json"
        }

device_ids = [4, 5, 6, 7, 8, 9, 10, 11]

signal_ids = ['signal-33']

df_irrad_aquiraz = extract_data(headers, device_ids, signal_ids, '2025-01-01 00:00:00')

df_irrad_aquiraz.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1438 entries, 0 to 1437
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   tempo                     1438 non-null   object 
 1   Aquiraz - UFV 01 - INV01  731 non-null    float64
 2   Aquiraz - UFV 03 - INV01  731 non-null    float64
 3   Aquiraz - UFV 01 - INV03  731 non-null    float64
 4   Aquiraz - UFV 02 - INV03  731 non-null    float64
 5   Aquiraz - UFV 03 - INV02  731 non-null    float64
 6   Aquiraz - UFV 02 - INV02  731 non-null    float64
 7   Aquiraz - UFV 02 - INV01  731 non-null    float64
 8   Aquiraz - UFV 01 - INV02  731 non-null    float64
dtypes: float64(8), object(1)
memory usage: 101.2+ KB


In [9]:
# Listar Solar Fields (device_type_id=10)
solar_fields = get_devices_by_type(10, api_key)
display(solar_fields)

Tentativa 1: api-key = UfKVCjs9h9QvsMhsSTNg...
✅ Sucesso com formato: api-key


[{'id': 1, 'name': 'Aquiraz - UFV 01'},
 {'id': 2, 'name': 'Aquiraz - UFV 03'},
 {'id': 3, 'name': 'Aquiraz - UFV 02'},
 {'id': 204, 'name': 'Unitêxtil - UFV 01'},
 {'id': 205, 'name': 'Unitêxtil - UFV 14'},
 {'id': 206, 'name': 'Unitêxtil - UFV 13'},
 {'id': 207, 'name': 'Unitêxtil - UFV 12'},
 {'id': 208, 'name': 'Unitêxtil - UFV 07'},
 {'id': 209, 'name': 'Unitêxtil - UFV 06'},
 {'id': 210, 'name': 'Unitêxtil - UFV 10'},
 {'id': 211, 'name': 'Unitêxtil - UFV 11'},
 {'id': 212, 'name': 'Unitêxtil - UFV 08'},
 {'id': 213, 'name': 'Unitêxtil - UFV 09'},
 {'id': 214, 'name': 'Unitêxtil - UFV 04'},
 {'id': 215, 'name': 'Unitêxtil - UFV 05'},
 {'id': 216, 'name': 'Unitêxtil - UFV 02'},
 {'id': 217, 'name': 'Unitêxtil - UFV 03'},
 {'id': 522, 'name': 'Massapê II'},
 {'id': 796, 'name': 'Beberibe'},
 {'id': 1033, 'name': 'Várzea Alegre 1'},
 {'id': 1034, 'name': 'Várzea Alegre 2'},
 {'id': 1035, 'name': 'Itapipoca'}]

In [ ]:
headers = {
            "API-Key": "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP",
            "Content-Type": "application/json"
        }
api_key = "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP"
# Listar Solar Fields (device_type_id=10) e extrair dados para todos os IDs
solar_fields = get_devices_by_type(10, api_key)

device_ids_sf = [sf['id'] for sf in solar_fields]

dfs_irradiancia_sf = {}

for sf_id in device_ids_sf:
    try:
        df = extract_data(
            headers,
            device_ids=[sf_id],
            variable_ids=['signal-39'],  # Tilted Irradiance - Solar Field
            device_type_id=10,
            start_time='2025-01-01 00:00:00',
            end_time='2026-03-16 20:00:00'
        )
        dfs_irradiancia_sf[sf_id] = df
        print(f"ID {sf_id}: {df.shape}")
    except Exception as e:
        print(f"ID {sf_id}: erro - {e}")

# Exportar para Excel com uma aba por solar field
output_path = 'irradiancia_solar_fields.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for sf_id, df in dfs_irradiancia_sf.items():
        df.to_excel(writer, sheet_name=f'SF_{sf_id}', index=False)

print(f"Exportado para {output_path}")


Tentativa 1: api-key = UfKVCjs9h9QvsMhsSTNg...
✅ Sucesso com formato: api-key
ID 1: (42582, 2)
ID 2: (42582, 2)
ID 3: (42582, 2)
ID 204: (44117, 2)


'❌ Erro na requisição: Status code 502, Detalhes: {\n  "message":"An invalid response was received from the upstream server",\n  "request_id":"25b4ec70666daa7506c2d0366f78040c"\n}'

ID 205: (43666, 2)


'❌ Erro na requisição: Status code 502, Detalhes: {\n  "message":"An invalid response was received from the upstream server",\n  "request_id":"57569631f1840d04c75cd82ca7687e3d"\n}'

ID 206: (43461, 2)
ID 207: (44496, 2)
ID 208: (44117, 2)
ID 209: (44117, 2)
ID 210: (44496, 2)
ID 211: (44496, 2)


'❌ Erro na requisição: Status code 502, Detalhes: {\n  "message":"An invalid response was received from the upstream server",\n  "request_id":"831b273316c85b507ee57c09ef83ae65"\n}'

ID 212: (43242, 2)
ID 213: (44496, 2)
ID 214: (44117, 2)
ID 215: (44117, 2)
ID 216: (44117, 2)
ID 217: (43255, 2)
ID 522: (44394, 2)
ID 796: (38177, 2)
ID 1033: (12031, 2)
ID 1034: (13738, 2)
ID 1035: (41533, 2)
Exportado para irradiancia_solar_fields.xlsx


In [1]:
{
 'id': 33,
 'name': 'Tilted Irradiance',
 'device_type_id': 2,
 'device_type': 'Inverter',
 'source_id': 4,
 'source': 'Calc - Physical Model',
 'description': 'Irradiância inclinada média do inversor. Essa variável é associada conforme a irradiância inclinada do solar field',
 'unit': 'W/m²',
 'data_type': 'double precision'
}

{'id': 33,
 'name': 'Tilted Irradiance',
 'device_type_id': 2,
 'device_type': 'Inverter',
 'source_id': 4,
 'source': 'Calc - Physical Model',
 'description': 'Irradiância inclinada média do inversor. Essa variável é associada conforme a irradiância inclinada do solar field',
 'unit': 'W/m²',
 'data_type': 'double precision'}

In [9]:
import requests

headers = {
            "API-Key": "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP",
            "Content-Type": "application/json"
        }
api_key = "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP"
# Listar Inverters (device_type_id=2) e extrair dados para todos os IDs
inverters = get_devices_by_type(2, api_key)

device_ids_inverter = [inverter['id'] for inverter in inverters]

dfs_irradiancia_inverters = {}

for inverter_id in device_ids_inverter:
    try:
        df = extract_data(
            headers,
            device_ids=[inverter_id],
            variable_ids=['signal-33'],  # Tilted Irradiance - Inverter
            device_type_id=2,
            start_time='2026-04-01 00:00:00',
            end_time='2026-05-06 23:59:00'
        )

        dfs_irradiancia_inverters[inverter_id] = df
        print(f"ID {inverter_id}: {df.shape}")
    except Exception as e:
        print(f"ID {inverter_id}: erro - {e}")

# Exportar para Excel com uma aba por inverter
output_path = 'irradiancia_inverters_teste.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for inverter_id, df in dfs_irradiancia_inverters.items():
        df.to_excel(writer, sheet_name=f'Inverter_{inverter_id}', index=False)

print(f"Exportado para {output_path}")


Tentativa 1: api-key = UfKVCjs9h9QvsMhsSTNg...
✅ Sucesso com formato: api-key


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 4: (2829, 2)


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 5: (2829, 2)


...

ID 1677: (3289, 2)


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 1678: (3289, 2)


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 1679: (3289, 2)
Exportado para irradiancia_inverters_teste.xlsx


In [1]:
{
 'id': 34,
 'name': 'Tilted Irradiance - Temperature Corrected',
 'device_type_id': 2,
 'device_type': 'Inverter',
 'source_id': 4,
 'source': 'Calc - Physical Model',
 'description': 'Irradiância inclinada corrigida pela temperatura dos módulos',
 'unit': 'W/m²',
 'data_type': 'double precision'
}

{'id': 34,
 'name': 'Tilted Irradiance - Temperature Corrected',
 'device_type_id': 2,
 'device_type': 'Inverter',
 'source_id': 4,
 'source': 'Calc - Physical Model',
 'description': 'Irradiância inclinada corrigida pela temperatura dos módulos',
 'unit': 'W/m²',
 'data_type': 'double precision'}

In [ ]:
headers = {
            "API-Key": "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP",
            "Content-Type": "application/json"
        }
api_key = "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP"
# Listar Inverters (device_type_id=2) e extrair dados para todos os IDs
inverters = get_devices_by_type(2, api_key)

device_ids_inverter = [inverter['id'] for inverter in inverters]

dfs_irradiancia_inverters = {}

for inverter_id in device_ids_inverter:
    try:
        df = extract_data(
            headers,
            device_ids=[inverter_id],
            variable_ids=['signal-34'],  # Tilted Irradiance - Inverter
            device_type_id=2,
            start_time='2025-01-01 00:00:00',
            end_time='2026-03-16 20:00:00'
        )
        dfs_irradiancia_inverters[inverter_id] = df
        print(f"ID {inverter_id}: {df.shape}")
    except Exception as e:
        print(f"ID {inverter_id}: erro - {e}")

# Exportar para Excel com uma aba por inverter
output_path = 'irradiancia_inverters.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for inverter_id, df in dfs_irradiancia_inverters.items():
        df.to_excel(writer, sheet_name=f'Inverter_{inverter_id}', index=False)

print(f"Exportado para {output_path}")


In [9]:
{
 'id': 45,
 'name': 'Inverter Tilted Irradiance',
 'device_type_id': 6,
 'device_type': 'String',
 'source_id': 3,
 'source': 'Calculated',
 'description': None,
 'unit': '-',
 'data_type': 'double precision'
}

{'id': 45,
 'name': 'Inverter Tilted Irradiance',
 'device_type_id': 6,
 'device_type': 'String',
 'source_id': 3,
 'source': 'Calculated',
 'description': None,
 'unit': '-',
 'data_type': 'double precision'}

In [ ]:
headers = {
            "API-Key": "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP",
            "Content-Type": "application/json"
        }
api_key = "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP"
# Listar Inverters (device_type_id=6) e extrair dados para todos os IDs
inverters = get_devices_by_type(6, api_key)

device_ids_inverter = [inverter['id'] for inverter in inverters if inverter['name'].startswith('Aquiraz - UFV 01')]

dfs_irradiancia_inverters = {}

for inverter_id in device_ids_inverter:
    try:
        df = extract_data(
            headers,
            device_ids=[inverter_id],
            variable_ids=['signal-45'],  # Tilted Irradiance - Inverter
            device_type_id=6,
            start_time='2025-01-01 00:00:00',
            end_time='2026-03-16 20:00:00'
        )
        dfs_irradiancia_inverters[inverter_id] = df
        print(f"ID {inverter_id}: {df.shape}")
    except Exception as e:
        print(f"ID {inverter_id}: erro - {e}")

# Exportar para Excel com uma aba por inverter
output_path = 'inverters_tilted_irradiance.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for inverter_id, df in dfs_irradiancia_inverters.items():
        df.to_excel(writer, sheet_name=f'Inverter_{inverter_id}', index=False)

print(f"Exportado para {output_path}")


Tentativa 1: api-key = UfKVCjs9h9QvsMhsSTNg...
✅ Sucesso com formato: api-key
ID 12: (42126, 2)
ID 13: (42126, 2)
ID 15: (42126, 2)
ID 16: (42126, 2)
ID 17: (42126, 2)
ID 18: (42126, 2)
ID 19: (42126, 2)
ID 20: (42126, 2)
ID 22: (42126, 2)
ID 23: (42126, 2)
ID 24: (42126, 2)
ID 25: (42126, 2)
ID 26: (42126, 2)
ID 27: (42126, 2)
ID 28: (42126, 2)
ID 29: (42126, 2)
ID 31: (42126, 2)
ID 32: (42126, 2)
ID 33: (42126, 2)
ID 34: (42126, 2)
ID 36: (42126, 2)
ID 38: (42126, 2)
ID 39: (42126, 2)
ID 40: (42126, 2)
ID 41: (42126, 2)
ID 42: (42126, 2)
ID 44: (42126, 2)
ID 45: (42126, 2)
ID 46: (42126, 2)
ID 47: (42126, 2)
ID 50: (42126, 2)
ID 51: (42126, 2)
ID 53: (42126, 2)
ID 56: (42126, 2)
ID 58: (42126, 2)
ID 60: (42126, 2)
ID 61: (42126, 2)
ID 62: (42126, 2)
ID 67: (42126, 2)
ID 68: (42126, 2)
ID 71: (42126, 2)
ID 73: (42126, 2)
ID 75: (42126, 2)
ID 77: (42126, 2)
ID 79: (42126, 2)
ID 80: (42126, 2)
ID 82: (42126, 2)
ID 83: (42126, 2)
ID 85: (42126, 2)
ID 87: (42126, 2)
ID 89: (42126, 2)
ID 9

KeyboardInterrupt: 

In [ ]:
{
 'id': 39,
 'name': 'Tilted Irradiance',
 'device_type_id': 10,
 'device_type': 'Solar Field',
 'source_id': 4,
 'source': 'Calc - Physical Model',
 'description': 'Irradiância inclinada média na superfície do solar field',
 'unit': 'W/m²',
 'data_type': 'double precision'
}

In [ ]:
headers = {
            "API-Key": "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP",
            "Content-Type": "application/json"
        }
api_key = "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP"
# Listar Inverters (device_type_id=6) e extrair dados para todos os IDs
inverters = get_devices_by_type(10, api_key)

device_ids_inverter = [inverter['id'] for inverter in inverters]

dfs_irradiancia_inverters = {}

for inverter_id in device_ids_inverter:
    try:
        df = extract_data(
            headers,
            device_ids=[inverter_id],
            variable_ids=['signal-39'],  # Tilted Irradiance
            device_type_id=10,
            start_time='2025-01-01 00:00:00',
            end_time='2026-05-16 20:00:00'
        )
        dfs_irradiancia_inverters[inverter_id] = df
        print(f"ID {inverter_id}: {df.shape}")
    except Exception as e:
        print(f"ID {inverter_id}: erro - {e}")

# Exportar para Excel com uma aba por inverter
output_path = 'inverters_tilted_irradiance.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for inverter_id, df in dfs_irradiancia_inverters.items():
        df.to_excel(writer, sheet_name=f'Inverter_{inverter_id}', index=False)

print(f"Exportado para {output_path}")


In [ ]:
{
    "id": 36,
    "name": "Active Power From Inverter",
    "device_type_id": 10,
    "device_type": "Solar Field",
    "source_id": 3,
    "source": "Calculated",
    "description": "Potência Ativa total da UFV, calculada pelo somatório das potências ativas dos inversores",
    "unit": "kW",
    "data_type": "double precision"
}

In [14]:
headers = {
            "API-Key": "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP",
            "Content-Type": "application/json"
        }
api_key = "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP"
# Listar Inverters (device_type_id=6) e extrair dados para todos os IDs
inverters = get_devices_by_type(10, api_key)

device_ids_inverter = [inverter['id'] for inverter in inverters]

dfs_irradiancia_inverters = {}

for inverter_id in device_ids_inverter:
    try:
        df = extract_data(
            headers,
            device_ids=[inverter_id],
            variable_ids=['signal-36'],  # Active Power From Inverter
            device_type_id=10,
            start_time='2025-01-01 00:00:00',
            end_time='2026-05-16 20:00:00'
        )
        dfs_irradiancia_inverters[inverter_id] = df
        print(f"ID {inverter_id}: {df.shape}")
    except Exception as e:
        print(f"ID {inverter_id}: erro - {e}")

# Exportar para Excel com uma aba por inverter
output_path = 'inverters_tilted_irradiance.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for inverter_id, df in dfs_irradiancia_inverters.items():
        df.to_excel(writer, sheet_name=f'Inverter_{inverter_id}', index=False)

print(f"Exportado para {output_path}")


Tentativa 1: api-key = UfKVCjs9h9QvsMhsSTNg...
✅ Sucesso com formato: api-key


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 1: (65794, 2)


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 2: (66017, 2)


...

ID 1033: (47007, 2)


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 1034: (51439, 2)


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 1035: (47936, 2)
Exportado para inverters_tilted_irradiance.xlsx


In [15]:
temp_ambiente = {
    "id": 48,
    "name": "Ambient Temperature",
    "device_type_id": 10,
    "device_type": "Solar Field",
    "source_id": 3,
    "source": "Calculated",
    "description": "Temperatura ambiente do solar field a uma altura de 2 metros",
    "unit": "°C",
    "data_type": "double precision"
}

inverters = get_devices_by_type(10, api_key)

device_ids_inverter = [inverter['id'] for inverter in inverters]

dfs_irradiancia_inverters = {}

for inverter_id in device_ids_inverter:
    try:
        df = extract_data(
            headers,
            device_ids=[inverter_id],
            variable_ids=['signal-48'],  # Ambient Temperature
            device_type_id=10,
            start_time='2025-01-01 00:00:00',
            end_time='2026-05-16 20:00:00'
        )
        dfs_irradiancia_inverters[inverter_id] = df
        print(f"ID {inverter_id}: {df.shape}")
    except Exception as e:
        print(f"ID {inverter_id}: erro - {e}")

# Exportar para Excel com uma aba por inverter
output_path = 'inverters_ambient_temperature.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for inverter_id, df in dfs_irradiancia_inverters.items():
        df.to_excel(writer, sheet_name=f'Inverter_{inverter_id}', index=False)

print(f"Exportado para {output_path}")

Tentativa 1: api-key = UfKVCjs9h9QvsMhsSTNg...
✅ Sucesso com formato: api-key


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 1: (50236, 2)


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 2: (50236, 2)


...

ID 1033: (17419, 2)


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 1034: (19130, 2)


C:\Users\usuario\AppData\Local\Temp\ipykernel_14820\865071597.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(batch_list, ignore_index=True)


ID 1035: (7460, 2)
Exportado para inverters_ambient_temperature.xlsx


In [10]:
temp_ambiente = {
    "id": 136,
    "name": "Internal Temperature",
    "device_type_id": 2,
    "device_type": "Inverter",
    "source_id": 1,
    "source": "SCADA",
    "description": "",
    "unit": "°C",
    "data_type": "double precision"
}

headers = {
            "API-Key": "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP",
            "Content-Type": "application/json"
        }
api_key = "UfKVCjs9h9QvsMhsSTNg2kmPDc5qycmP"

inverters = get_devices_by_type(2, api_key)

device_ids_inverter = [inverter['id'] for inverter in inverters]

dfs_irradiancia_inverters = {}

for inverter_id in device_ids_inverter:
    try:
        df = extract_data(
            headers,
            device_ids=[inverter_id],
            variable_ids=['signal-136'],  # Internal Temperature inverter
            device_type_id=2,
            start_time='2025-01-01 00:00:00',
            end_time='2026-05-16 20:00:00'
        )
        dfs_irradiancia_inverters[inverter_id] = df
        print(f"ID {inverter_id}: {df.shape}")
    except Exception as e:
        print(f"ID {inverter_id}: erro - {e}")

# Exportar para Excel com uma aba por inverter
output_path = 'inverters_internal_inverter_temperature.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for inverter_id, df in dfs_irradiancia_inverters.items():
        df.to_excel(writer, sheet_name=f'Inverter_{inverter_id}', index=False)

print(f"Exportado para {output_path}")

Tentativa 1: api-key = UfKVCjs9h9QvsMhsSTNg...
✅ Sucesso com formato: api-key
ID 4: (0, 2)
ID 5: (0, 2)
ID 6: (0, 2)
ID 7: (0, 2)
ID 8: (0, 2)
ID 9: (0, 2)
ID 10: (0, 2)
ID 11: (0, 2)
ID 218: (0, 2)
ID 219: (0, 2)
ID 220: (0, 2)
ID 221: (0, 2)
ID 222: (0, 2)
ID 223: (0, 2)
ID 224: (0, 2)
ID 225: (0, 2)
ID 226: (0, 2)
ID 227: (0, 2)
ID 228: (0, 2)
ID 229: (0, 2)
ID 230: (0, 2)
ID 231: (0, 2)
ID 232: (0, 2)
ID 233: (0, 2)
ID 523: (0, 2)
ID 524: (0, 2)
ID 525: (0, 2)
ID 526: (0, 2)
ID 527: (0, 2)
ID 528: (0, 2)
ID 529: (0, 2)
ID 530: (0, 2)
ID 531: (0, 2)
ID 532: (0, 2)
ID 533: (0, 2)
ID 534: (0, 2)
ID 535: (0, 2)
ID 803: (0, 2)
ID 804: (0, 2)
ID 805: (0, 2)
ID 806: (0, 2)
ID 807: (0, 2)
ID 808: (0, 2)
ID 809: (0, 2)
ID 810: (0, 2)
ID 811: (0, 2)


'❌ Erro na requisição: Status code 502, Detalhes: Bad Gateway'

ID 812: (0, 2)
ID 1036: (0, 2)
ID 1037: (0, 2)
ID 1038: (0, 2)
ID 1039: (0, 2)
ID 1040: (0, 2)
ID 1041: (0, 2)
ID 1042: (0, 2)
ID 1043: (0, 2)
ID 1044: (0, 2)
ID 1045: (0, 2)
ID 1046: (0, 2)
ID 1047: (0, 2)
ID 1048: (0, 2)
ID 1049: (0, 2)
ID 1050: (0, 2)
ID 1051: (0, 2)
ID 1052: (0, 2)
ID 1053: (0, 2)
ID 1054: (0, 2)
ID 1055: (0, 2)
ID 1057: (0, 2)
ID 1058: (0, 2)
ID 1059: (0, 2)
ID 1060: (0, 2)
ID 1061: (0, 2)
ID 1062: (0, 2)
ID 1063: (0, 2)
ID 1064: (0, 2)
ID 1656: (0, 2)
ID 1657: (0, 2)
ID 1658: (0, 2)
ID 1659: (0, 2)
ID 1660: (0, 2)
ID 1661: (0, 2)
ID 1662: (0, 2)
ID 1663: (0, 2)
ID 1664: (0, 2)
ID 1665: (0, 2)
ID 1666: (0, 2)
ID 1667: (0, 2)
ID 1668: (0, 2)
ID 1669: (0, 2)
ID 1670: (0, 2)
ID 1671: (0, 2)
ID 1672: (0, 2)
ID 1673: (0, 2)
ID 1674: (0, 2)
ID 1675: (0, 2)
ID 1676: (0, 2)


'❌ Erro na requisição: Status code 502, Detalhes: {\n  "message":"An invalid response was received from the upstream server",\n  "request_id":"73ede53c58654b40d132dd2a69ad077a"\n}'

ID 1677: (0, 2)
ID 1678: (0, 2)
ID 1679: (0, 2)
Exportado para inverters_internal_inverter_temperature.xlsx


In [13]:
temp_ambiente = {
    "id": 302,
    "name": "Wind Direction",
    "device_type_id": 3,
    "device_type": "Meteo",
    "source_id": 1,
    "source": "SCADA",
    "description": "",
    "unit": "deg",
    "data_type": "double precision"
  }

inverters = get_devices_by_type(3, api_key)

device_ids_inverter = [inverter['id'] for inverter in inverters]

dfs_irradiancia_inverters = {}

for inverter_id in device_ids_inverter:
    try:
        df = extract_data(
            headers,
            device_ids=[inverter_id],
            variable_ids=['signal-302'],  # Wind Direction
            device_type_id=3,
            start_time='2025-01-01 00:00:00',
            end_time='2026-05-16 20:00:00'
        )
        dfs_irradiancia_inverters[inverter_id] = df
        print(f"ID {inverter_id}: {df.shape}")
    except Exception as e:
        print(f"ID {inverter_id}: erro - {e}")

# Exportar para Excel com uma aba por inverter
output_path = 'inverters_internal_inverter_temperature.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for inverter_id, df in dfs_irradiancia_inverters.items():
        df.to_excel(writer, sheet_name=f'Inverter_{inverter_id}', index=False)

print(f"Exportado para {output_path}")

Tentativa 1: api-key = UfKVCjs9h9QvsMhsSTNg...
✅ Sucesso com formato: api-key
ID 798: (0, 2)
ID 1630: (0, 2)


'❌ Erro na requisição: Status code 500, Detalhes: Internal Server Error'

'❌ Erro na requisição: Status code 500, Detalhes: Internal Server Error'

'❌ Erro na requisição: Status code 500, Detalhes: Internal Server Error'

'❌ Erro na requisição: Status code 500, Detalhes: Internal Server Error'

...

'❌ Erro na requisição: Status code 500, Detalhes: Internal Server Error'

'❌ Erro na requisição: Status code 500, Detalhes: Internal Server Error'

'❌ Erro na requisição: Status code 500, Detalhes: Internal Server Error'

'❌ Erro na requisição: Status code 500, Detalhes: Internal Server Error'

ID 2163: erro - Status code: 500, Erro: Internal Server Error
Exportado para inverters_internal_inverter_temperature.xlsx
